# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-morad15/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
from huggingface_hub import login
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)

con = duckdb.connect()

REL_FEB = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
"""

REL_MAR = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Setup ready.")

Setup ready.


In [2]:
baseline_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
    FROM {REL_FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS march_avg_position
    FROM {REL_MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.*,
    m.march_avg_position,

    CASE
        WHEN m.march_avg_position > f.gsc_avg_position
        THEN 1
        ELSE 0
    END AS review_priority

FROM feb f
INNER JOIN march m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE f.gsc_avg_position IS NOT NULL
  AND m.march_avg_position IS NOT NULL
""").df()

print("Modeling dataset:", baseline_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling dataset: (134238, 9)


In [3]:
print("Available variables:")
for name in sorted(globals()):
    if not name.startswith("_"):
        obj = globals()[name]
        if hasattr(obj, "shape") or "model" in name.lower() or "pred" in name.lower() or "score" in name.lower():
            print(name, type(obj).__name__, getattr(obj, "shape", ""))

Available variables:
baseline_df DataFrame (134238, 9)
np module <function shape at 0x7b84c076a660>


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


The action queue ranks content items for human review using observable performance signals from the validated feature set.

The ranking is a decision-support aid, not an automatic recommendation to edit or remove a page. Higher-ranked items should be reviewed first because they combine measurable visibility or engagement signals with a potential opportunity for improvement.

Reason codes are designed to remain understandable and traceable:

* **HIGH_VISIBILITY** — the content already has measurable search impressions.
* **CLICK_OPPORTUNITY** — the content has measurable impressions and clicks, making click performance reviewable.
* **POSITION_REVIEW** — the page has measurable search visibility and its average position can be reviewed.
* **ENGAGEMENT_REVIEW** — session or engaged-session signals are available for additional context.
* **PRIORITY_REVIEW** — the item receives a high composite review-priority score from the observable signals.

The queue is intended to help a human decide where to look first. It does not establish causality and does not guarantee that an intervention will improve future search performance.


In [4]:
import numpy as np
import pandas as pd

# Work on a copy so the source dataframe is not modified.
queue_df = baseline_df.copy()

# Fill missing engagement/session values with 0 for ranking purposes.
# The missingness itself is retained as a separate audit signal.
queue_df["ga4_sessions_missing"] = queue_df["ga4_sessions"].isna()
queue_df["ga4_engaged_sessions_missing"] = queue_df["ga4_engaged_sessions"].isna()

queue_df["ga4_sessions"] = queue_df["ga4_sessions"].fillna(0)
queue_df["ga4_engaged_sessions"] = queue_df["ga4_engaged_sessions"].fillna(0)

# Rank each observable signal using percentile rank.
def percentile_rank(series):
    return series.rank(pct=True, method="average")

queue_df["impressions_rank"] = percentile_rank(queue_df["gsc_impressions"])
queue_df["clicks_rank"] = percentile_rank(queue_df["gsc_clicks"])
queue_df["position_rank"] = percentile_rank(
    -queue_df["gsc_avg_position"]
)
queue_df["sessions_rank"] = percentile_rank(queue_df["ga4_sessions"])
queue_df["engaged_sessions_rank"] = percentile_rank(
    queue_df["ga4_engaged_sessions"]
)

# Transparent composite review score.
queue_df["review_score"] = (
    0.30 * queue_df["impressions_rank"]
    + 0.20 * queue_df["clicks_rank"]
    + 0.20 * queue_df["position_rank"]
    + 0.15 * queue_df["sessions_rank"]
    + 0.15 * queue_df["engaged_sessions_rank"]
)

# Generate human-readable reason codes.
def make_reason_codes(row):
    reasons = []

    if row["gsc_impressions"] > 0:
        reasons.append("HIGH_VISIBILITY")

    if row["gsc_impressions"] > 0 and row["gsc_clicks"] > 0:
        reasons.append("CLICK_OPPORTUNITY")

    if row["gsc_impressions"] > 0 and row["gsc_avg_position"] <= 20:
        reasons.append("POSITION_REVIEW")

    if row["ga4_sessions"] > 0 or row["ga4_engaged_sessions"] > 0:
        reasons.append("ENGAGEMENT_REVIEW")

    if not reasons:
        reasons.append("LOW_SIGNAL")

    return "|".join(reasons)

queue_df["reason_codes"] = queue_df.apply(make_reason_codes, axis=1)

# Final ranking.
queue_df = queue_df.sort_values(
    "review_score",
    ascending=False
).reset_index(drop=True)

queue_df["review_rank"] = np.arange(1, len(queue_df) + 1)

# Basic validation checks.
assert len(queue_df) == len(baseline_df)
assert queue_df["review_rank"].is_unique
assert queue_df["review_score"].notna().all()

print("Action queue created successfully.")
print("Rows:", len(queue_df))
print("Top review score:", round(queue_df["review_score"].iloc[0], 4))
print("\nTop 10 action candidates:")

display(
    queue_df[
        [
            "review_rank",
            "review_score",
            "reason_codes",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_sessions",
            "ga4_engaged_sessions",
        ]
    ].head(10)
)

Action queue created successfully.
Rows: 134238
Top review score: 0.9893

Top 10 action candidates:


,review_rank,review_score,reason_codes,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,1,0.989306,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,26801.0,190.0,1.614147,86.0,13.0
1,2,0.989214,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,142215.0,1605.0,1.882000,395.0,14.0
2,3,0.987837,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,51836.0,324.0,1.965819,154.0,10.0
3,4,0.984905,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,14632.0,60.0,1.421467,62.0,5.0
4,5,0.984061,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,35520.0,314.0,2.275810,336.0,37.0
5,6,0.983798,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,83657.0,52.0,1.402359,49.0,1.0
6,7,0.983558,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,21970.0,96.0,1.924429,55.0,4.0
7,8,0.983014,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,19436.0,94.0,2.028557,91.0,11.0
8,9,0.982779,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,19224.0,73.0,1.992889,132.0,4.0
9,10,0.982354,HIGH_VISIBILITY|CLICK_OPPORTUNITY|POSITION_REV...,43693.0,166.0,2.328759,88.0,6.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


The action queue is intended for content-review prioritization.

A human reviewer can use the ranked queue to decide which content items to inspect first, using the reason codes and underlying search and engagement signals as supporting evidence.

The output is valid as a directional decision-support tool for the same type of content data and feature definitions used during validation. It should not be treated as a production ranking system or as a prediction of future Google performance.

Important limits:

* The ranking describes observed relationships in the available dataset; it does not establish causation.
* A high review score does not mean that an edit will improve impressions, clicks, or rankings.
* The queue should not be used to automatically publish, rewrite, delete, or redirect content.
* Missing GA4 session values are treated as zero for the composite ranking, so those rows require additional human attention before interpretation.
* Changes in data definitions, traffic mix, search behavior, or feature availability may make the ranking less reliable.
* The current output is a research prototype and should be re-evaluated before any production use.


In [5]:
# Validate the intended-use boundaries and document the queue coverage.

print("Intended-use validation")
print("-" * 40)

print(f"Queue rows: {len(queue_df):,}")
print(f"Unique review ranks: {queue_df['review_rank'].nunique():,}")
print(
    f"Rows with missing GA4 sessions: "
    f"{queue_df['ga4_sessions_missing'].sum():,}"
)
print(
    f"Rows with missing GA4 engaged sessions: "
    f"{queue_df['ga4_engaged_sessions_missing'].sum():,}"
)

# Check that the ranking contains no invalid scores.
assert queue_df["review_score"].between(0, 1).all()
assert queue_df["review_rank"].min() == 1
assert queue_df["review_rank"].max() == len(queue_df)

print("\nChecks passed.")
print("The queue is suitable for directional human-review prioritization.")

Intended-use validation
----------------------------------------
Queue rows: 134,238
Unique review ranks: 134,238
Rows with missing GA4 sessions: 67,969
Rows with missing GA4 engaged sessions: 67,969

Checks passed.
The queue is suitable for directional human-review prioritization.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


Every ranked item requires human review before any content action is taken.

The reviewer should check:

1. **Search evidence** — confirm that impressions, clicks, and average position are meaningful for the page.
2. **Content quality** — check whether the page is accurate, useful, complete, and aligned with the search intent.
3. **Freshness** — check whether facts, statistics, examples, references, or recommendations are outdated.
4. **Search intent** — confirm that the page is targeting the right intent and that the content satisfies the user's likely need.
5. **Cannibalization** — check whether another page covers the same demand before creating or expanding content.
6. **Business context** — consider strategic importance, conversions, and other information that is not represented in the model features.
7. **Data quality** — investigate missing or unusual metrics before relying on the ranking.

### No-go cases

The system should **not** automatically:

* Publish or rewrite content.
* Delete, redirect, or merge pages.
* Change titles, URLs, or search targeting without review.
* Declare a page successful or unsuccessful from the ranking alone.
* Treat the ranking as a Google ranking prediction.
* Make medical, legal, financial, or other high-stakes content decisions without qualified human review.
* Use missing GA4 values as evidence that engagement is poor.
* Treat a reason code as proof of the underlying cause.

The model output is therefore a prioritization aid. The final decision remains with a human reviewer who can use information outside the modeled feature set.


In [6]:
# Human-review and no-go audit checks.

required_columns = [
    "review_rank",
    "review_score",
    "reason_codes",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
]

missing_columns = [
    col for col in required_columns
    if col not in queue_df.columns
]

assert not missing_columns, (
    f"Missing required queue columns: {missing_columns}"
)

# Count items requiring additional data-quality attention.
data_quality_review = (
    queue_df["ga4_sessions_missing"]
    | queue_df["ga4_engaged_sessions_missing"]
)

print("Human-review audit")
print("-" * 40)
print(f"Total queue items: {len(queue_df):,}")
print(f"Items requiring GA4 data-quality review: {data_quality_review.sum():,}")
print(
    f"Items with complete GA4 fields: "
    f"{(~data_quality_review).sum():,}"
)

# Confirm that no action decision is being generated automatically.
automatic_action_columns = [
    col for col in queue_df.columns
    if col.lower() in {
        "publish",
        "delete",
        "redirect",
        "rewrite",
        "auto_action"
    }
]

assert not automatic_action_columns, (
    f"Unexpected automatic action columns found: "
    f"{automatic_action_columns}"
)

print("\nChecks passed.")
print("No automatic content action is produced by the queue.")
print("Human review remains required before intervention.")

Human-review audit
----------------------------------------
Total queue items: 134,238
Items requiring GA4 data-quality review: 67,969
Items with complete GA4 fields: 66,269

Checks passed.
No automatic content action is produced by the queue.
Human review remains required before intervention.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


The recommendations should be treated as directional decision-support, not a permanent production rule.

Monitoring should focus on whether the ranking remains useful over time. A refresh or retraining review should be considered if:

* The distribution of review scores changes substantially from the validation period.
* The share of items with missing GA4 fields increases materially.
* Precision@50 or another agreed ranking metric declines on newly labeled data.
* The relationship between the input features and `review_priority` changes.
* The content mix or traffic patterns change enough that the current ranking may no longer represent the same decision problem.

Because this workflow is non-production, these are review triggers rather than automatic retraining rules. A human should inspect the change, confirm the data quality, and decide whether the model or thresholds need to be refreshed.


In [10]:
# Find DataFrame variables currently available

dataframes = {
    name: obj.shape
    for name, obj in globals().items()
    if hasattr(obj, "shape") and hasattr(obj, "columns")
}

print("Available DataFrames:")
for name, shape in dataframes.items():
    print(f"- {name}: {shape}")

Available DataFrames:
- baseline_df: (134238, 9)
- queue_df: (134238, 19)


In [13]:
#  Monitoring / retrain trigger checks

print("Monitoring / retrain trigger checks")
print("-" * 40)

# Current queue size and score distribution
print(f"Queue rows: {len(queue_df):,}")
print(f"Mean review score: {queue_df['review_score'].mean():.4f}")
print(f"Median review score: {queue_df['review_score'].median():.4f}")
print(f"Minimum review score: {queue_df['review_score'].min():.4f}")
print(f"Maximum review score: {queue_df['review_score'].max():.4f}")

# Missing-data monitoring
ga4_missing = queue_df["ga4_sessions"].isna().mean()
engaged_missing = queue_df["ga4_engaged_sessions"].isna().mean()

print(f"Missing GA4 sessions rate: {ga4_missing:.2%}")
print(f"Missing GA4 engaged sessions rate: {engaged_missing:.2%}")

# Top-50 concentration check
top_50 = queue_df.head(50)

print(f"\nTop-50 mean review score: {top_50['review_score'].mean():.4f}")
print(f"Top-50 minimum review score: {top_50['review_score'].min():.4f}")

print("\nMonitoring checks completed.")
print("These values are monitoring baselines, not automatic retrain decisions.")

Monitoring / retrain trigger checks
----------------------------------------
Queue rows: 134,238
Mean review score: 0.5000
Median review score: 0.4661
Minimum review score: 0.2002
Maximum review score: 0.9893
Missing GA4 sessions rate: 0.00%
Missing GA4 engaged sessions rate: 0.00%

Top-50 mean review score: 0.9772
Top-50 minimum review score: 0.9697

Monitoring checks completed.
These values are monitoring baselines, not automatic retrain decisions.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


The ranked action queue is exported so that the paper can reuse the same decision-support output without changing the ranking.

The export contains the review rank, review score, reason codes, and supporting feature values used for human review. The queue is regenerated by the notebook rather than committed as a data file.

A small summary of the ranking distribution is also exported as a JSON receipt for traceability.


In [14]:
# Exports for the paper

from pathlib import Path
import json

# Create output directory
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export ranked queue
queue_path = output_dir / "ranked_action_queue.csv"
queue_df.to_csv(queue_path, index=False)

# Export summary metrics / receipt
summary = {
    "queue_rows": int(len(queue_df)),
    "unique_review_ranks": int(queue_df["review_rank"].nunique()),
    "top_review_score": float(queue_df["review_score"].max()),
    "top_50_mean_review_score": float(queue_df.head(50)["review_score"].mean()),
    "top_50_min_review_score": float(queue_df.head(50)["review_score"].min()),
    "missing_ga4_sessions": int(queue_df["ga4_sessions"].isna().sum()),
    "missing_ga4_engaged_sessions": int(
        queue_df["ga4_engaged_sessions"].isna().sum()
    ),
    "human_review_required": True,
    "automatic_content_action": False
}

summary_path = output_dir / "action_playbook_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Exports created successfully.")
print(f"Queue export: {queue_path}")
print(f"Summary export: {summary_path}")
print()
print("Queue rows:", len(queue_df))
print("Top review score:", f"{queue_df['review_score'].max():.4f}")
print("Top-50 mean score:", f"{queue_df.head(50)['review_score'].mean():.4f}")

Exports created successfully.
Queue export: ../outputs/ranked_action_queue.csv
Summary export: ../outputs/action_playbook_summary.json

Queue rows: 134238
Top review score: 0.9893
Top-50 mean score: 0.9772


In [15]:
# Self-check

checks = {
    "queue_created": len(queue_df) > 0,
    "review_rank_unique": queue_df["review_rank"].nunique() == len(queue_df),
    "review_score_present": queue_df["review_score"].notna().all(),
    "reason_codes_present": queue_df["reason_codes"].notna().all(),
    "queue_exported": queue_path.exists(),
    "summary_exported": summary_path.exists(),
    "human_review_required": summary["human_review_required"] is True,
    "no_automatic_action": summary["automatic_content_action"] is False,
}

print("Self-check")
print("-" * 40)

for check, passed in checks.items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

print("-" * 40)

if all(checks.values()):
    print("All self-checks passed.")
else:
    raise AssertionError("One or more self-checks failed.")

Self-check
----------------------------------------
queue_created: PASS
review_rank_unique: PASS
review_score_present: PASS
reason_codes_present: PASS
queue_exported: PASS
summary_exported: PASS
human_review_required: PASS
no_automatic_action: PASS
----------------------------------------
All self-checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.